# Week 2: Natrual Language Processing and Word Embeddings

Natural language processing with deep learning is a powerful combination. Using word vector representations and embedding layers, train recurrent neural networks with outstanding performance across a wide variety of applications, including sentiment analysis, named entity recognition and neural machine translation.

**Learning Objectives:**

- Explain how word embeddings capture relationships between words
- Load pre-trained word vectors
- Measure similarity between word vectors using cosine similarity
- Use word embeddings to solve word analogy problems such as Man is to Woman as King is to ______.
- Reduce bias in word embeddings
- Create an embedding layer in Keras with pre-trained word vectors
- Describe how negative sampling learns word vectors more efficiently than other methods
- Explain the advantages and disadvantages of the GloVe algorithm
- Build a sentiment classifier using word embeddings
- Build and train a more sophisticated classifier using an LSTM

---

## Table of Contents

---

## Word Representation

This section introduces a fundamental shift in how Deep Learning models process language, moving from simple, isolated word counts to rich, "featurized" representations. The core message is that **Word Embeddings** allow algorithms to generalize knowledge by understanding that certain words (like "apple" and "orange") are semantically similar.

### Key Takeaways

* **The Limitations of One-Hot Vectors:**
    * Previously, words were represented as vectors with a single "1" in a specific position (e.g., $O_{5391}$ for "man"). 
    * **The Mathematical Flaw:** The inner product between any two different one-hot vectors is always $0$. This means the model mathematically "sees" no more similarity between "apple" and "orange" than it does between "apple" and "king."
* **Featurized Representations (The "Why"):** 
    * By assigning a high-dimensional vector (e.g., 300 dimensions) to each word, we can represent characteristics like **Gender, Royalty, Age,** or **Food status**.
    * For instance, "man" might have a gender value of $-1$, while "woman" has $+1$. "Apple" and "orange" would both have high values for the "Food" feature but neutral values for "Royalty."

<div style='display: flex; justify-content: center'>
    <img src='images/word_embeddings.png' width=750px>
</div>


* **Generalization & Analogies:** 
    * Embeddings enable models to understand analogies ($man:woman :: king:queen$).
    * Crucially, if a model learns that "orange juice" is a common phrase, the shared features in the embedding space allow it to automatically infer that "apple juice" is also a likely valid phrase.

* **Visualizing High Dimensions:** 
    * Since we cannot visualize 300D space, algorithms like **t-SNE** are used to map these embeddings into 2D plots. 
    * In these plots, related concepts (animals, fruits, numbers) naturally cluster together, proving the model has "learned" semantic relationships.

<div style='display: flex; justify-content: center'>
    <img src='images/word_embeddings_visualization.png' width=500px>
</div>

* **Addressing Bias:** * Since these models learn from real-world data, they can pick up gender or ethnicity biases. A key part of the modern NLP workflow is **debiasing** these embeddings to ensure fairer outcomes.

### One-Hot vs. Embeddings: A Comparison

| Feature | One-Hot Encoding | Word Embeddings |
| :--- | :--- | :--- |
| **Sparsity** | Sparse (mostly zeros) | Dense (filled with floats) |
| **Vector Size** | Vocabulary size (e.g., 10,000) | Fixed size (e.g., 300) |
| **Similarity** | $0$ for all word pairs | Higher for related words |
| **Generalization** | Impossible | Built-in |

---

## Using Word Embeddings

This section highlights how **Word Embeddings** act as a catalyst for **Transfer Learning** in NLP. The core idea is that by learning word relationships from massive, unlabeled datasets, we can significantly improve performance on specific tasks where labeled data is scarce.

### High-level Summary

The primary utility of word embeddings is their ability to bridge the gap between a model's lack of world knowledge and the specific requirements of an NLP task. 

Let's consider the following two sentences:
1. Sally Robinson is an orange farmer.
2. Robert Lin is a durian cultivator.

In a typical scenario, a model might struggle to identify "Robert Lin" as a person's name if it follows the phrase "durian cultivator"—simply because those words are rare. However, because word embeddings are "pre-trained" on nearly the entire internet, the model has already learned that a durian is a fruit (like an orange) and a cultivator is a profession (like a farmer). By swapping **one-hot vectors** for these **dense embeddings**, we "transfer" that massive internet-scale knowledge into our small, local project.

This process is fundamentally a form of transfer learning: we leverage a "Task A" (learning word relationships from billions of unlabeled sentences) to solve a "Task B" (labeling names in a few thousand sentences). While similar to **Face Encodings** in vision, the "vocabulary" approach in NLP makes word embeddings uniquely efficient for processing text with a fixed dictionary.


### Key Takeaways

* **Generalization through Semantic Similarity:**
    * Embeddings allow models to recognize that "durian cultivator" is conceptually similar to "orange farmer," even if the specific words "durian" or "cultivator" were never seen during the task's training phase.
* **The Three-Step Transfer Learning Process:**
    1.  **Pre-training:** Learn embeddings from a massive unlabeled corpus (1B to 100B words) or download pre-trained vectors.
    2.  **Transfer:** Apply these dense, low-dimensional vectors (e.g., 300D) to a target task with a small labeled dataset.
    3.  **Fine-tuning (Optional):** Adjust the embeddings during target task training only if the labeled dataset is sufficiently large.
* **Dimensionality Advantage:**
    * Replaces sparse, 10,000-dimensional **one-hot vectors** with dense, 300-dimensional **embedding vectors**, making the input representation much more efficient and meaningful.
* **Task Suitability:**
    * Most beneficial for tasks with limited data like **Named Entity Recognition (NER)**, text summarization, and parsing.
    * Less critical for data-rich tasks like Machine Translation or Language Modeling.
* **Embeddings vs. Face Encodings:**
    * **Similarity:** Both represent complex data as fixed-length continuous vectors (encodings).
    * **Difference:** Face recognition requires a system that can encode *any new face*, whereas word embeddings typically map a *fixed vocabulary* of words to learned vectors.

---

## Properties of Word Embeddings

This section explores the fascinating ability of **Word Embeddings** to perform **Analogy Reasoning** using simple vector arithmetic. By representing words in a high-dimensional space (e.g., 300D), the model captures semantic relationships as geometric distances and directions.

### High-level Summary

The beauty of word embeddings lies in their mathematical "common sense." By analyzing billions of words, these models realize that the leap from "man" to "woman" is spatially almost identical to the leap from "king" to "queen." It isn't just memorizing definitions; it is mapping the **structure of language** onto a coordinate system.

When we ask the model an analogy, we are essentially asking it to complete a parallelogram in a 300-dimensional forest. To find the missing corner, we use **Cosine Similarity**, which focuses on the direction the vectors are pointing rather than just their raw length. This allows the model to correctly identify everything from geographical capitals to grammatical patterns, providing a powerful intuition that these "dense vectors" truly capture the features of the objects they represent.

### Key Takeaways

* **Vector Arithmetic for Analogies:**
    * The core discovery (by [Mikolov et al.](https://scottyih.org/files/rvecs.pdf)) is that the difference between word vectors often represents a specific semantic concept.
    * **Example:** $e_{man} - e_{woman} \approx e_{king} - e_{queen}$. In both cases, the resulting vector represents the concept of "gender."
* **The Analogy Algorithm:**
    * To solve "Man is to Woman as King is to ??", the algorithm searches for a word $w$ that satisfies the equation:
        $$e_{man} - e_{woman} \approx e_{king} - e_w$$
    * Mathematically, this is solved by finding a word $w$ that maximizes the similarity to the vector: $e_{king} - e_{man} + e_{woman}$.

<div style='display: flex; justify-content: center'>
    <img src='images/analogies_with_word_embed.png' width=750px>
</div>

* **Cosine Similarity:**
    * The most common function used to measure how "close" two word vectors are is **Cosine Similarity**:
        $$\text{Similarity}(u, v) = \frac{u^T v}{||u||_2 ||v||_2}$$
    * This measures the cosine of the angle $\phi$ between vectors. If the angle is $0^\circ$, the similarity is $1$; if they are orthogonal ($90^\circ$), it is $0$.
    * Another similarity measure is negative value of square differences $||\vec{u}  -\vec{v}||^2$. This is a less-common approach since it does not normalize for embeddings norm.
* **Broad Generalization:**
    * Embeddings can learn diverse relationships beyond gender, such as:
        * **Verb Tense:** $Big : Bigger :: Tall : Taller$
        * **Capitals:** $Ottawa : Canada :: Nairobi : Kenya$
        * **Currency:** $Yen : Japan :: Ruble : Russia$
* **Visualization Warning:**
    * While we use **t-SNE** to visualize embeddings in 2D, t-SNE is a **non-linear** mapping. You should not expect the "parallelogram" relationship of analogies to hold true in a t-SNE plot; it only reliably exists in the original high-dimensional space.

---

## Embedding Matrix

In this section, the focus shifts from the *why* of word embeddings to the *how*—specifically, the data structures used to store and retrieve them. The core concept introduced is the **Embedding Matrix ($E$)**, which acts as a lookup table for every word in a vocabulary.

### High-level Summary

Think of the **Embedding Matrix** as a giant library where every column is a book containing the "personality" of a single word. If our vocabulary has 10,000 words, we have 10,000 columns. 

When a model needs to "look up" a word like "Orange," it uses a **One-Hot Vector** as a pointer. Mathematically, when we multiply the Embedding Matrix $E$ by this pointer, the zeros in the one-hot vector cancel out every single word in the dictionary except for the one we want. The result is a clean, 300-dimensional vector that represents "Orange" in a way the neural network can actually understand.

While we represent this as matrix multiplication in theory, in practice, it’s much more efficient: software frameworks like TensorFlow or PyTorch use specialized **"Embedding Layers"** that perform a direct index lookup (essentially just grabbing the column) to save on computation, since multiplying by a vector of zeros is technically a waste of energy!

### Key Takeaways

* **The Embedding Matrix ($E$):**
    * This is a massive matrix of shape $(d, V)$, where $d$ is the embedding dimension (e.g., **300**) and $V$ is the vocabulary size (e.g., **10,000**).
    * Each column in this matrix represents the "featurized" 300-dimensional vector for a specific word.
* **The One-Hot Vector ($O$):**
    * To find a specific word, we use a one-hot vector $O_j$, which is a $V \times 1$ vector containing a **1** at index $j$ and **0**s elsewhere.
* **Retrieval via Matrix Multiplication:**
    * Mathematically, selecting an embedding is represented as the product: $e_j = E \cdot O_j$.
    * Since the one-hot vector has only a single "1," this operation effectively "filters out" all other columns and returns only the 300-dimensional column corresponding to that specific word.
* **Notation:**
    * The resulting 300-dimensional vector is denoted as $e_j$ (or $e_{orange}$ in the example), representing the dense embedding for the $j$-th word in the vocabulary.

---

## Learning Word Embeddings

This section marks the transition from the "what" of embeddings to the "how" of the algorithms that create them. It explains that while early methods were complex neural language models, researchers discovered that much simpler tasks—like predicting a word from its neighbor—can produce equally powerful word embeddings.

### High-level Summary

The history of word embeddings is a journey toward simplicity. Initially, researchers built full-scale neural networks to predict the next word in a sentence. To do this, the network had to "squish" words into a 300-dimensional space. To get the answer right consistently, the network realized it was easier if it treated similar concepts (like different fruits) as being in the same "neighborhood" of that 300D space. This byproduct—the neighborhood map—is the **Embedding Matrix**.

As the field evolved, researchers realized they didn't need a complex window of four or six words to learn these relationships. They discovered that we can learn an excellent "map" of language by simply picking one random word (the context) and trying to guess another word that appears nearby (the target). This led to the creation of the **Skip-Gram** and **Word2Vec** models, which are far more computationally efficient and are the industry standards we use today.

### Key Takeaways

* **Learning via Language Modeling:**
    * The earliest successful embedding algorithms (like the one by [Bengio et al.](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)) were designed to solve a **Language Modeling** task: predicting the next word in a sequence (e.g., "I want a glass of orange [?]").
* **The Neural Architecture:**
    * **Input:** A fixed "historical window" of words (e.g., the previous 4 words).
    * **Embedding Layer:** Each word is converted from a one-hot vector to a 300D dense vector using a shared embedding matrix $E$.
    * **Hidden Layer:** The 4 embedding vectors are concatenated (e.g., into a 1,200D vector) and passed through a standard neural network layer.
    * **Output:** A **Softmax** layer that predicts the probability of all 10,000 words in the vocabulary being the "target" word.

<div style='Display: flex; Justify-content: center'>
    <img src='images/neural_lm.png' width=750px>
</div>

* **The Incentive for Similarity:**
    * To minimize loss, the model is incentivized to give "apple" and "orange" similar embeddings. This is because both words often appear before "juice," and having similar vectors allows the model to use the same internal weights to predict the correct next word.
* **Alternative Contexts for Learning:**
    * If the goal is specifically to learn embeddings (rather than a perfect language model), researchers found they could vary the **"Context"** used to predict the **"Target"**:
        * **Words on Left and Right:** Predicting the middle word from surrounding words.
        * **Last One Word:** Predicting the next word given only the immediate predecessor.
        * **Nearby One Word (Skip-Gram):** Predicting a word that appears anywhere nearby, even if it's not the immediate next word.